In [5]:
import yfinance as yf
import pandas as pd
import numpy as np
from statsmodels.tsa.stattools import coint

# 1. FETCH DATA (Downloading data up to today - October 2026!)
df1 = yf.download("V", start="2019-01-01", end="2026-10-14", progress=False)
df2 = yf.download("MA", start="2019-01-01", end="2026-10-14", progress=False)

# Safe column extraction
if isinstance(df1.columns, pd.MultiIndex):
    close_V = df1['Close']['V'] if 'V' in df1['Close'] else df1['Close'].iloc[:, 0]
    close_MA = df2['Close']['MA'] if 'MA' in df2['Close'] else df2['Close'].iloc[:, 0]
else:
    close_V = df1['Close']
    close_MA = df2['Close']

data = pd.DataFrame({'V': close_V, 'MA': close_MA}).dropna()

# Split into 3 periods!
# Training: until end of 2021
# Test 1 (Main result for CV): 2022 - October 2024
# Test 2 (Stress Test for recruiter): October 2024 - Today (October 2026)
train_data = data[:"2021-12-31"]
test1_data = data["2022-01-01":"2024-10-01"] 
test2_data = data["2024-10-02":]

price_X_train = train_data['V'].to_numpy()
price_Y_train = train_data['MA'].to_numpy()

# 2. PROPER COINTEGRATION TEST (ONLY ON TRAINING SET!)
coint_t, p_value, _ = coint(price_X_train, price_Y_train)
print(f"Cointegration test p-value (Training set 2019-2021 only): {p_value:.4f}")

# Blind Beta estimation
covariance_matrix = np.cov(price_X_train, price_Y_train)
beta = covariance_matrix[0, 1] / covariance_matrix[0, 0]


def run_backtest(price_X, price_Y, beta_val, window=60, threshold=2.0, cost_bps=0.001):
    spread = price_Y - beta_val * price_X
    spread_series = pd.Series(spread)
    
    rolling_mean = spread_series.rolling(window=window).mean().shift(1)
    rolling_std = spread_series.rolling(window=window).std().shift(1)
    
    signals = np.zeros(len(spread))
    current_pos = 0
    
    for i in range(window, len(spread)):
        if rolling_std.iloc[i] == 0 or pd.isna(rolling_std.iloc[i]): continue
        z_score = (spread[i] - rolling_mean.iloc[i]) / rolling_std.iloc[i]
        
        if current_pos == 0:
            if z_score > threshold: current_pos = -1
            elif z_score < -threshold: current_pos = 1
        elif current_pos == 1 and z_score >= 0: current_pos = 0
        elif current_pos == -1 and z_score <= 0: current_pos = 0
            
        signals[i] = current_pos
        
    signals_series = pd.Series(signals)
    capital_invested = price_Y + np.abs(beta_val) * price_X
    spread_diff = np.insert(np.diff(spread), 0, 0)
    
    raw_returns = (spread_diff / np.insert(capital_invested[:-1], 0, capital_invested[0])) * signals_series.shift(1).fillna(0)
    trade_events = np.abs(signals_series.diff().fillna(0))
    costs_array = trade_events * cost_bps
    
    strategy_returns = (raw_returns - costs_array).iloc[window:].dropna()
    
    num_trades = (trade_events.iloc[window:] > 0).sum()
    time_in_market = (signals_series.iloc[window:] != 0).mean()
    
    ann_return = strategy_returns.mean() * 252
    sharpe = (strategy_returns.mean() / strategy_returns.std()) * np.sqrt(252) if strategy_returns.std() != 0 else 0
    
    cum_ret = (1 + strategy_returns).cumprod()
    max_dd = ((cum_ret - cum_ret.cummax()) / cum_ret.cummax()).min()
    
    return ann_return, sharpe, max_dd, int(num_trades), time_in_market

# 3. TEST 1 RESULTS (2022 to October 2024)
ann1, sh1, dd1, trades1, tim1 = run_backtest(test1_data['V'].to_numpy(), test1_data['MA'].to_numpy(), beta)
print("\n--- TEST 1 (Jan 2022 - Oct 2024) ---")
print(f"Annualized Return: {ann1 * 100:.2f}% | Sharpe: {sh1:.2f} | Max DD: {dd1 * 100:.2f}%")
print(f"Number of Trades (N): {trades1} | Time in Market: {tim1 * 100:.1f}%")

# 4. TEST 2 RESULTS (Stress Test: Since October 2024 to Today - October 2026)
ann2, sh2, dd2, trades2, tim2 = run_backtest(test2_data['V'].to_numpy(), test2_data['MA'].to_numpy(), beta)
print("\n--- TEST 2 (Regime Break Stress Test: Oct 2024 - Oct 2026) ---")
print(f"Annualized Return: {ann2 * 100:.2f}% | Sharpe: {sh2:.2f} | Max DD: {dd2 * 100:.2f}%")
print(f"Number of Trades (N): {trades2} | Time in Market: {tim2 * 100:.1f}%")

Cointegration test p-value (Training set 2019-2021 only): 0.1302

--- TEST 1 (Jan 2022 - Oct 2024) ---
Annualized Return: 3.54% | Sharpe: 0.89 | Max DD: -2.34%
Number of Trades (N): 25 | Time in Market: 64.3%

--- TEST 2 (Regime Break Stress Test: Oct 2024 - Oct 2026) ---
Annualized Return: 1.74% | Sharpe: 0.48 | Max DD: -3.34%
Number of Trades (N): 18 | Time in Market: 46.3%
